In [1]:
#http://127.0.0.1:8888/?token=641ff58c730676d8f40d01ea1af56b3dc8e8be248d9aeb4a
import os
import re
import operator
import matplotlib.pyplot as plt
import warnings
import gensim
import numpy as np
warnings.filterwarnings('ignore')  # Let's not pay heed to them right now

from gensim.models import CoherenceModel, LdaModel, LsiModel, HdpModel
from gensim.models.wrappers import LdaMallet
from gensim.corpora import Dictionary
from pprint import pprint
import json
import pyLDAvis.gensim
import datetime
from nltk.stem.lancaster import LancasterStemmer
from nltk.corpus import stopwords
from nltk.stem.porter import *
import numpy as np
import string
%matplotlib inline
import pandas as pd

c:\users\stefa\pycharmprojects\tesi_nlp\venv\lib\site-packages\nltk\decorators.py:68: DeprecationWarning: `formatargspec` is deprecated since Python 3.5. Use `signature` and the `Signature` object directly
  regargs, varargs, varkwargs, defaults, formatvalue=lambda value: ""


In [2]:
# import file to calculate LDA
# REDDIT
filename = 'Vettel_messages_2019_september_data.csv'
df = pd.read_csv(os.path.join(os.getcwd(), 'reddit_data', 'clean_data', filename))
print(len(df))
# TELEGRAM
# filename = '_Telegram_cleanedData_2019-09-01_2019-09-17.csv'
# df = pd.read_csv(os.path.join(os.getcwd(), 'telegram_data', 'clean_data', filename))
# df = df.iloc[:,1:]
# print(df.columns)

FileNotFoundError: File b'C:\\Users\\stefa\\PycharmProjects\\tesi_nlp\\reddit_data\\clean_data\\Vettel_messages_2019_september_data.csv' does not exist

In [69]:
df.columns

Index(['Unnamed: 0', 'subreddit', 'body', 'created_utc', 'author'], dtype='object')

In [61]:
# Select only data desidered:
# df = df[df.author == 'mindy2000']
#df1 = df[df.subreddit == 'Futurology']
print(len(df))

15921


In [ ]:
def build_texts(fname):
    """
    Function to build tokenized texts from file
    fname: File to be read
    Returns: yields preprocessed line
    """
    
    for line in fname:
        yield gensim.utils.simple_preprocess(line, deacc=True, min_len=3)
            

def clean_text(text):
    try:
        stop_words = set(stopwords.words('english'))
        #porter = PorterStemmer()
        #lemman = LancasterStemmer()
        translator = str.maketrans('', '', string.punctuation)
        text = re.sub(r'http\S+', '', text)
        text = text.translate(translator)
        return ' '.join([w for w in text.split() if w not in stop_words])
        #return ' '.join([lemman.stem(porter.stem(w)) for w in text.split() if w not in stop_words])

    except:
        return ''


In [71]:
# Clean text in dataframe
cleaned_data = [clean_text(d) for d in df['body'].tolist()]
train_texts = list(build_texts(cleaned_data))
print(f'Number of messages --> {len(train_texts)}')

Number of messages --> 19855


In [72]:
dictionary = Dictionary(train_texts)
corpus = [dictionary.doc2bow(text) for text in train_texts]

In [73]:
import random
random.seed(113)
ldamodel = LdaModel(corpus=corpus, num_topics=5, id2word=dictionary)

In [74]:
pyLDAvis.enable_notebook()

In [75]:
pyLDAvis.gensim.prepare(ldamodel, corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.110368 -0.003244       1        1  34.851601
0     -0.066745  0.056723       2        1  32.409420
3     -0.045357  0.052181       3        1  17.752831
4      0.001108 -0.132502       4        1  10.519375
1      0.221361  0.026841       5        1   4.466776, topic_info=     Category          Freq          Term         Total  loglift  logprob
term                                                                     
39    Default  23501.000000        vettel  23501.000000  30.0000  30.0000
23    Default   9023.000000       leclerc   9023.000000  29.0000  29.0000
108   Default   4419.000000          race   4419.000000  28.0000  28.0000
990   Default    701.000000        inside    701.000000  27.0000  27.0000
6769  Default    924.000000     agreement    924.000000  26.0000  26.0000
177   Default    969.000000         track    969.000000  25.0000  25.0000
489   Default   3122.000000           car   3122.000000  24.0000  24.0000
608   Default   2550.000000        driver   2550.000000  23.0000  23.0000
26    Default    585.000000        racing    585.000000  22.0000  22.0000
315   Default   1191.000000           tow   1191.000000  21.0000  21.0000
1004  Default    874.000000           pit    874.000000  20.0000  20.0000
96    Default   2980.000000          like   2980.000000  19.0000  19.0000
395   Default    639.000000          line    639.000000  18.0000  18.0000
35    Default   2174.000000           the   2174.000000  17.0000  17.0000
314   Default    753.000000    slipstream    753.000000  16.0000  16.0000
36    Default   1978.000000          time   1978.000000  15.0000  15.0000
228   Default   1610.000000           lap   1610.000000  14.0000  14.0000
403   Default    181.000000           per    181.000000  13.0000  13.0000
860   Default    529.000000        safety    529.000000  12.0000  12.0000
2537  Default    898.000000          swap    898.000000  11.0000  11.0000
778   Default    835.000000      undercut    835.000000  10.0000  10.0000
652   Default   3157.000000       charles   3157.000000   9.0000   9.0000
484   Default    721.000000           red    721.000000   8.0000   8.0000
988   Default    553.000000           air    553.000000   7.0000   7.0000
2767  Default    265.000000       penalty    265.000000   6.0000   6.0000
498   Default    788.000000           bad    788.000000   5.0000   5.0000
524   Default   1182.000000      strategy   1182.000000   4.0000   4.0000
127   Default   1276.000000          year   1276.000000   3.0000   3.0000
4751  Default    137.000000           pre    137.000000   2.0000   2.0000
756   Default    504.000000       fucking    504.000000   1.0000   1.0000
...       ...           ...           ...           ...      ...      ...
7434   Topic5     21.173845         della     21.976292   3.0713  -6.9101
9869   Topic5     21.008463          gara     21.811111   3.0710  -6.9179
1588   Topic5     89.188919       whining     94.641594   3.0492  -5.4721
6748   Topic5     27.333357      suggests     28.627131   3.0623  -6.6548
8562   Topic5     45.391434  conversation     48.963467   3.0328  -6.1475
2995   Topic5     92.924194           non    105.182747   2.9846  -5.4311
853    Topic5     47.904987        crofty     52.436253   3.0181  -6.0936
403    Topic5    135.919601           per    181.820389   2.8175  -5.0508
2182   Topic5     71.704536         rosso     92.298447   2.8560  -5.6903
2183   Topic5     63.374092          toro     80.596573   2.8681  -5.8138
445    Topic5     35.161716         tenth     39.950054   2.9808  -6.4029
639    Topic5     32.867252        silver     36.982933   2.9905  -6.4704
3986   Topic5     41.191425      tomorrow     52.551582   2.8649  -6.2446
3463   Topic5     32.337654       podiums     39.326462   2.9128  -6.4866
513    Topic5     61.564159      incident    138.941208   2.2945  -5.8428
39     Topic5    646.941345